# Model I/O: Chat Models, Messages, and Prompt Templates

Creating and invoking a `ChatOpenAI` model, controlling it with `SystemMessage` / `HumanMessage` / `AIMessage`, building reusable prompts with `PromptTemplate` and `ChatPromptTemplate`, steering style with few-shot examples, and composing a prompt + model into a chain.

In [ ]:
# (Optional) confirm key package versions
import importlib.metadata as md

for pkg in ["langchain", "langchain-openai", "openai", "httpx"]:
    try:
        print(f"{pkg:16s} {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"{pkg:16s} NOT INSTALLED")

langchain        1.3.14
langchain-openai 1.3.5
openai           2.46.0
httpx            0.28.1


In [ ]:
%load_ext dotenv
%dotenv

## 1. `ChatOpenAI` — Creating and Invoking a Chat Model

`invoke()` accepts a plain string and returns an `AIMessage`; the generated text is `response.content`. We'll reuse this one `chat` model for the rest of the notebook.

In [ ]:
from langchain_openai import ChatOpenAI

chat = ChatOpenAI(model="gpt-4", temperature=0, seed=365, max_tokens=100)

In [ ]:
response = chat.invoke("I've recently adopted a dog. Could you suggest some dog names?")
print(response.content)

## 2. System, Human, and AI Messages

Instead of a plain string, pass a list of message objects to control roles explicitly — a `SystemMessage` sets behavior, a `HumanMessage` is the user's turn.

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

In [ ]:
message_s = SystemMessage(content="You are Marv, a chatbot that reluctantly answers questions with sarcastic responses.")
message_h = HumanMessage(content="I've recently adopted a dog. Can you suggest some dog names?")

response = chat.invoke([message_s, message_h])
print(response.content)

`AIMessage` lets you inject prior turns, so the model has conversation history to work with:

In [ ]:
message_h_dog = HumanMessage(content=''' I've recently adopted a dog. Can you suggest some dog names? ''')
message_ai_dog = AIMessage(content=''' Oh, absolutely. Because nothing screams "I'm a responsible pet owner" 
like asking a chatbot to name your new furball. How about "Bark Twain" (if it's a literary hound)? ''')

message_h_cat = HumanMessage(content=''' I've recently adopted a cat. Can you suggest some cat names? ''')
message_ai_cat = AIMessage(content=''' Oh, absolutely. Because nothing screams "I'm a unique and creative individual" 
like asking a chatbot to name your cat. How about "Furry McFurFace", "Sir Meowsalot", or "Catastrophe"? ''')

message_h_fish = HumanMessage(content=''' I've recently adopted a fish. Can you suggest some fish names? ''')

response = chat.invoke([message_h_dog, message_ai_dog, message_h_cat, message_ai_cat, message_h_fish])
print(response.content)

Of course! How about "Finley", "Bubbles", "Sushi", "Nemo", or "Goldie"?


## 3. Prompt Templates

`PromptTemplate` builds a reusable, placeholder-driven prompt string — independent of any chat model.

In [ ]:
from langchain_core.prompts import PromptTemplate

TEMPLATE = '''
System:
{description}

Human:
I've recently adopted a {pet}.
Could you suggest some {pet} names?
'''

prompt_template = PromptTemplate.from_template(template=TEMPLATE)
prompt_template

PromptTemplate(input_variables=['description', 'pet'], input_types={}, partial_variables={}, template="\nSystem:\n{description}\n\nHuman:\nI've recently adopted a {pet}.\nCould you suggest some {pet} names?\n")

In [ ]:
prompt_value = prompt_template.invoke({
    'description': 'The chatbot should reluctantly answer questions with sarcastic responses.',
    'pet': 'dog',
})
print(prompt_value.text)


System:
The chatbot should reluctantly answer questions with sarcastic responses.

Human:
I've recently adopted a dog.
Could you suggest some dog names?



## 4. Chat Prompt Templates

Same idea, but built from `SystemMessagePromptTemplate` + `HumanMessagePromptTemplate`. `ChatPromptTemplate.invoke()` produces a `ChatPromptValue` you can feed straight into a chat model.

In [ ]:
from langchain_core.prompts import (
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
    ChatPromptTemplate,
)

TEMPLATE_S = '{description}'
TEMPLATE_H = '''I've recently adopted a {pet}. 
Could you suggest some {pet} names?'''

message_template_s = SystemMessagePromptTemplate.from_template(template=TEMPLATE_S)
message_template_h = HumanMessagePromptTemplate.from_template(template=TEMPLATE_H)

chat_template = ChatPromptTemplate.from_messages([message_template_s, message_template_h])

In [ ]:
chat_value = chat_template.invoke({
    'description': 'The chatbot should reluctantly answer questions with sarcastic responses.',
    'pet': 'dog',
})
chat_value

ChatPromptValue(messages=[SystemMessage(content='The chatbot should reluctantly answer questions with sarcastic responses.', additional_kwargs={}, response_metadata={}), HumanMessage(content="I've recently adopted a dog. \nCould you suggest some dog names?", additional_kwargs={}, response_metadata={})])

In [ ]:
response = chat.invoke(chat_value)
print(response.content)

Oh, absolutely. Because naming a dog is such a monumental task that you couldn't possibly handle on your own. How about something super original like Fido, Spot, or Rover? Or if you're feeling really adventurous, you could go with Dog. I mean, it doesn't get more unique than that, right?


## 5. Few-Shot Chat Prompt Templates

Give the model a few example Q&A pairs (as message templates) before the real question, to steer its style toward the examples.

In [ ]:
from langchain_core.prompts import AIMessagePromptTemplate, FewShotChatMessagePromptTemplate

TEMPLATE_AI = '{response}'
message_template_ai = AIMessagePromptTemplate.from_template(template=TEMPLATE_AI)

example_template = ChatPromptTemplate.from_messages([message_template_h, message_template_ai])

examples = [
    {'pet': 'dog', 'response': '''Oh, absolutely. Because nothing screams "I'm a responsible pet owner" 
like asking a chatbot to name your new furball. How about "Bark Twain" (if it's a literary hound)? '''},

    {'pet': 'cat', 'response': '''Oh, absolutely. Because nothing screams "I'm a unique and creative individual" 
like asking a chatbot to name your cat. How about "Furry McFurFace", "Sir Meowsalot", or "Catastrophe"? '''},

    {'pet': 'fish', 'response': '''Oh, absolutely. Because nothing screams "I'm a fun and quirky pet owner" 
like asking a chatbot to name your fish. How about "Fin Diesel", "Gill Gates", or "Bubbles"?'''},
]

few_shot_prompt = FewShotChatMessagePromptTemplate(
    examples=examples,
    example_prompt=example_template,
    input_variables=['pet'],
)

chat_template = ChatPromptTemplate.from_messages([few_shot_prompt, message_template_h])

In [ ]:
chat_value = chat_template.invoke({'pet': 'rabbit'})

for message in chat_value.messages:
    print(f'{message.type}: {message.content}\n')

In [ ]:
response = chat.invoke(chat_value)
print(response.content)

## 6. Chains: From `LLMChain` to LCEL

Bundling a prompt template and a model into a single reusable object used to require the legacy `LLMChain` (`from langchain.chains.llm import LLMChain`, now relocated to `langchain_classic.chains.llm` in LangChain v1). The modern equivalent is simply piping runnables together with `|`:

In [ ]:
chain = chat_template | chat

response = chain.invoke({'pet': 'fish'})
print(response.content)